<p style="text-align: right; margin: 0; line-height: 0;"><img width="100px" height="auto" src="../../.admin/img/eth_logo.png"></p>

<h1 style="text-align: center;">05 Mobile AI</h1>
<hr>


## **4. Getting Started with Raspberry Pi**

<div class="alert alert-light">

  **Name:** (add your name here)

  **Matriculation Number:** (add your number here)

</div>

### 4.1 Introduction

Your Raspberry Pi runs language models locally, without a cloud service, an account or an internet connection. Everything you send to it stays on the small computer in front of you.

In this notebook you connect the Pi to your laptop, check that the two can talk, and learn how the setup works. Then you meet Ollama, the program on the Pi that runs the models, and ask your first questions from Python. The last part shows how to log in to the Pi yourself and watch it work while a model answers. Allow about **60 minutes**, plus time for the questions.

By the end, you should be able to connect to the Pi, explain which programs on it answer your requests, ask a model a question and read its statistics, and watch the Pi's processor and memory while a model runs.

#### 4.1.1 Environment Setup

This notebook runs on your laptop and talks to the Pi over the network cable. Section 4.7 also needs a terminal.

<div class="alert alert-block alert-success">

If a library is missing, use the installation instructions in [00.1_Check_Course_Environment.ipynb](../00_Intro/00.1_Check_Course_Environment.ipynb). Select that Python environment as this notebook's kernel.
</div>

### 4.2 The Raspberry Pi

We use a **Raspberry Pi 5 with 8 GB RAM** as the host machine for our local LLMs.

<center>
<img src="../../.admin/notebooks/picture/raspberrypi.png"
     alt="Raspberry Pi 5"
     width="60%"
     style="filter: grayscale(80%);">
</center>

The Raspberry Pi is a single-board computer: processor, memory, network and USB sit on one board the size of a credit card. Ours has four Arm Cortex-A76 cores at 2.4 GHz, 8 GB of memory, and a microSD card as its disk. It draws only a few watts, far less than a laptop, and its small built-in graphics processor does not help with the models (05.08 shows why).

That makes it a good stand-in for the devices this chapter is about. Phones, cars and sensors increasingly run models on the device itself, with similar processors and similar limits on memory and power. What runs well on the Pi gives you a feeling for what is possible at the edge.

The Pi runs a lightweight **headless** operating system, Raspberry Pi OS Lite, which is based on Debian Linux. A headless OS has no graphical user interface (GUI): you manage it remotely, over SSH or through an API. Without a desktop, more memory and processing power are left for the models. Servers, IoT devices and embedded systems are usually run this way.

**If you encounter any issues, do not hesitate to ask the teaching staff for assistance.**

### 4.3 How the Setup Works

#### 4.3.1 Connect power and laptop

The Raspberry Pi gets its power over USB-C. Use the official 27 W USB-C power supply and connect it to the USB-C port on the Pi. The LED lights up red, then green, and after around ten seconds it starts flickering. The flickering shows read and write activity on the SD card while the Pi starts up. After about a minute the light stays green, and the Pi is ready.

Connect the Pi to your laptop with Ethernet: either (1) the provided Ethernet cable, with a USB-C adapter if your laptop has no Ethernet port, or (2) the provided RJ45 to USB-C cable.

#### 4.3.2 The Pi's own small network

The cable between your laptop and the Pi forms a tiny network of two devices. The Pi always has the address `172.31.250.1`. It also runs a DHCP server, the same service your home router uses to hand out addresses, and gives your laptop an address between `172.31.250.100` and `172.31.250.200`. So your laptop needs no configuration, and every Pi in the course looks the same from your laptop.

The Pi offers your laptop an address, but no route to the internet (no "default gateway") and no DNS server. Your laptop therefore keeps using Wi-Fi for everything else, and your internet keeps working while the Pi is plugged in. The Pi itself has no internet connection in this setup.

#### 4.3.3 Two services on the Pi

Two programs on the Pi wait for requests from your laptop:

- **Ollama** on port `11434` runs the language models. Section 4.5 introduces it.
- A small **course server** on port `8000` reports the Pi's status and runs a few course programs.

Both speak HTTP, the protocol your browser uses for web pages. A request is a URL, sometimes with some JSON attached, and the answer is JSON.

```text
   your laptop  172.31.250.1xx               Raspberry Pi  172.31.250.1
  +----------------------------+            +-----------------------------+
  |  this notebook             |  Ethernet  |  Ollama         port 11434  |
  |  caspi.py  ---- HTTP ------+----------->|    the models               |
  |                            |            |  course server  port 8000   |
  |  Wi-Fi: your internet      |            |    status, scripts          |
  +----------------------------+            +-----------------------------+
```

You do not have to write these requests yourself. `caspi.py`, a short Python file next to this notebook, wraps them in functions such as `caspi.ask(...)`. It is plain Python with the `requests` library.

### 4.4 Reachability Tests

<div class="alert alert-block alert-info">
Quickly run the tests below to find out if the connection is working!</div>

We test in two steps: first one plain HTTP request, then the helper we use for the rest of the course.

#### 4.4.1 HTTP: is the course server answering?

The next cell asks the course server for its status report at `/health`, exactly the way `caspi.py` does it. If an answer comes back, the cable, the addresses and the programs on the Pi are all working.

In [ ]:
import requests

requests.get("http://172.31.250.1:8000/health", timeout=10).json()

The answer is a Python dictionary: the Pi's name, its mode, the image version, the processor temperature, the free disk space, whether Ollama is running, and which scripts the Pi offers.

#### 4.4.2 caspi: connect for the rest of the course

`caspi.connect()` asks both services and prints a short report. If you have just switched the Pi on, it keeps trying for a minute instead of failing immediately. Run it at the start of every notebook that uses the Pi.

In [ ]:
import caspi

caspi.connect()

<div class="alert alert-block alert-success">
If the cell above prints "Connected to cas-pi-...", everything in this course will work. Nothing else needs configuring.</div>

If it fails, the error message says what to check first. The list below covers the rest.

#### 4.4.3 Known issues (hopefully this section can be ignored)

<div class="alert alert-block alert-success">
Ignore these issues if you can connect to the Pi. Try to skip this part and continue with the next section.
</div>

Known issues are listed below. If you encounter any of these issues, or something unexpected happens, please reach out. We are happy to help.

- **The Pi is still booting:** the Pi needs about a minute after switch-on before it answers. Wait until the green light stops flickering and run the cell again.

- **Your laptop got no address:** if the cable was plugged in while the Pi was still booting, your laptop may give up asking the Pi for an address for a while. Windows then shows an address starting with `169.254`. Unplug the Ethernet cable at the laptop, wait a few seconds, and plug it in again.

- **VPN or firewall:** a VPN client or security software on your laptop can block the connection to the Pi, even when `ping 172.31.250.1` in a terminal works (Section 4.7.1). Disconnect the VPN while you work with the Pi.

- **macOS blocks the connection:** since macOS 15, an app may reach devices on your local network only with your permission. The first time VS Code talks to the Pi, macOS asks whether it may find and connect to devices on your local network: click **Allow**. If you clicked Don't Allow, `caspi.connect()` and `ping` in VS Code's terminal fail with "No route to host", although your Mac has an address from the Pi. Switch VS Code on under System Settings, Privacy & Security, Local Network (the same for iTerm or any other terminal app) and run the cell again.

- **IP address conflicts:** the Pi uses the subnet `172.31.250.0/24`, chosen because it is rare. If your home router happens to use the same subnet, the network can behave strangely while the Pi is plugged in. Connecting your laptop to a mobile hotspot, for example from your phone, while you use the Pi resolves the conflict.

- **Internet loss after connecting the Pi:** this should not happen, because the Pi does not offer itself as a gateway (Section 4.3.2). If it happens anyway, please tell us.

- **Missing tools:** the tools for this course are listed in [`00.1_Check_Course_Environment.ipynb`](../00_Intro/00.1_Check_Course_Environment.ipynb), which you worked through before this notebook.

### 4.5 Ollama

Ollama ([https://ollama.com](https://ollama.com/)) is an open-source program that runs open LLMs. It stores the model files, loads a model into memory when it is needed, runs it, and answers requests over its REST API on port 11434. The same API works on a laptop, a server or a Pi. Only the speed changes. For further information, check out the [official documentation](https://docs.ollama.com/api).

#### 4.5.1 The models on your Pi

Each Pi carries several models of different sizes.

In [ ]:
caspi.models(info=True)

- **size_gb** is the size of the model file on the SD card.
- **parameters** counts the model's weights, in millions (M) or billions (B).
- **quantization** says how each weight is stored. `Q4_K_M` uses about 4 to 5 bits per weight instead of 16, which makes the model roughly a third of its original size. Notebook 05.09 explains how this works.

Pick a chat model for the rest of this notebook. Not every model in the list can chat: `all-minilm` and `nomic-embed-text-v2-moe` only turn text into numbers (05.10), the `tev1` models only make decisions (05.14), and `qwen2.5-coder:1.5b-base` only completes code (05.17). We use `granite4.2:3b`: it loads in about a minute and a half and gives short, clean answers. `caspi.show()` prints what the model is.

In [ ]:
MODEL = "granite4.2:3b"   # any chat model from the list above
caspi.show(MODEL)

#### 4.5.2 Loading a model into memory

A model must sit in memory (RAM) to answer. Ollama loads it from the SD card at the first request, which takes between one and five minutes on the Pi, depending on the model's size. caspi asks Ollama to keep the model loaded for 30 minutes after each use, so only the first request is slow. `caspi.load()` does the loading now, so your first real question is not the slow one.

In [ ]:
caspi.load(MODEL)

`caspi.ps()` shows what is in memory right now. This is Ollama's own answer: `size` is the memory the model occupies in bytes, and `expires_at` is when Ollama will drop it if nobody uses it.

In [ ]:
caspi.ps()

The Pi has 8 GB of memory, and that limits how many models fit at once. Ollama keeps at most two models loaded. When you ask a third, it drops one of the others, so switching back and forth costs another load each time.

<div class="alert alert-block alert-warning">

**Watch the memory.** Two large models can already be too many, and Ollama does not always notice. When we loaded `qwen2.5-coder:7b` while `granite4.2:3b` and `qwen3.5:4b` were in memory, Ollama dropped one of them, but the coder and the other one still did not fit. The Pi ran out of memory, stopped answering and restarted by itself about five minutes later, with every model gone. A model with a file of 4 GB or more needs most of the memory on its own. Before you load one, or anything next to one, run `caspi.stop()`: without a model name, it drops every model and frees the memory. `caspi.connect()` also tells you when models from an earlier notebook are still loaded. Later notebooks remind you of this box where it matters.
</div>

### 4.6 Try It Out

#### 4.6.1 Your first question

The answer appears word by word, exactly as the model produces it. Granite normally thinks aloud before it answers. `think=False` switches that off for a short, direct reply. Notebook 05.13 looks at what thinking costs.

In [ ]:
reply = caspi.ask(MODEL, "Explain what a language model is, in three sentences, to a manager.", think=False)

After the answer, caspi prints a line of statistics that comes straight from Ollama:

- **prompt**: how many tokens the model read (your question plus some formatting), how many of them Ollama still had cached from an earlier request, and how fast it read the rest
- **reply**: how many tokens it wrote, and how fast
- **load**: seconds spent loading the model, close to zero if it was already in memory
- **total**: the whole request, from start to finish

Watch the reply speed over the course as you try other models.

#### 4.6.2 The reply as a Python object

`caspi.ask` also returns the answer, so you can keep working with it in Python. `reply.text` is the answer as a string, `reply.stats` holds the numbers from the line above, and `reply.raw` is Ollama's original last message, with its times in nanoseconds.

In [ ]:
words = len(reply.text.split())
print(words, "words,", reply.stats["reply_tokens"], "tokens")
reply.stats

#### 4.6.3 A conversation

A model remembers nothing between requests. To hold a conversation, we send the whole history every time: a list of messages, each with a role (`user` or `assistant`) and its content. `caspi.chat` takes such a list.

In [ ]:
history = [{"role": "user", "content": "Hi, I am Ana and I study mobile computing."}]
reply = caspi.chat(MODEL, history, think=False)

Append the answer and a follow-up question, then send everything again. Watch the prompt tokens in the statistics line: the whole conversation goes to the Pi once more. Ollama still has the start of the previous request in memory (the cached tokens), so it only has to read the new part.

In [ ]:
history.append(reply.message)
history.append({"role": "user", "content": "What is my name, and what do I study?"})
reply = caspi.chat(MODEL, history, think=False)

For comparison, `ask` sends only this one question:

In [ ]:
caspi.ask(MODEL, "What is my name, and what do I study?", think=False)

`caspi.repl` keeps the history for you, so you can simply type into the box under the cell and press Enter. `/reset` forgets the conversation, `/bye` ends it. The cell itself finishes right away, so you can run other cells while the chat stays open.

In [ ]:
caspi.repl(MODEL, think=False)

#### 4.6.4 The Pi itself

The course server can also run small programs on the Pi, and it streams their output back as it happens. `hello` is a test program that proves the whole path works.

In [ ]:
caspi.run("hello")

The status report from Section 4.4.1 is also available as `caspi.health()`. For example, the processor temperature in °C, after all the work above:

In [ ]:
caspi.health()["cpu_temp_c"]

#### 4.6.5 The cheat sheet

`caspi.help()` prints a one-page cheat sheet. Come back to it whenever you forget a command.

In [ ]:
caspi.help()

### 4.7 Manual Usage

Everything so far went through `caspi.py`. This section looks behind it: you log in to the Pi, read some of its files, use Ollama directly and watch the Pi work. You need a terminal: on Windows, PowerShell or Terminal; on macOS and Linux, Terminal.

#### 4.7.1 Log in with SSH

Open a terminal on your machine. First check that the Pi is on the network at all: `ping` sends a tiny packet to an address and waits for the echo. Windows sends four packets and stops; on macOS and Linux, press Ctrl+C to stop.

```sh
ping 172.31.250.1
```

Then log in. SSH (Secure Shell) is a network protocol used for remotely accessing other computers over a network:

```sh
ssh cas@172.31.250.1
```

When `ssh` connects to a device for the first time, a security warning shows up. Confirm that you want to continue connecting by typing `yes`. Then enter the password `password` (nothing appears while you type it). You now see a Linux shell, and everything you type runs on the Pi. The prompt `cas@cas-pi-...:~$` tells you that you are the user `cas` on your Pi.

<center>
<img src="../../.admin/notebooks/picture/raspberry_login.png"
     alt="Raspberry Pi 5 login"
     width="60%"
     style="filter: grayscale(10%);">
</center>

`exit` (or Ctrl+D) logs you out again. If `ssh` refuses with a warning that the remote host identification has changed, you probably used a different Pi at the same address before. Run `ssh-keygen -R 172.31.250.1` and try again.

The user `cas` may use `sudo`, so the Pi is yours to explore. If something breaks, tell us and we will reset the card.

#### 4.7.2 Look around with ls and cat

`ls` lists the files in a folder and `cat` prints a file. Each of these tells you something about the setup:

```sh
cat /etc/cas-release
cat /etc/systemd/system/ollama.service.d/10-cas.conf
ls /opt/cas/scripts
cat /opt/cas/scripts/hello
```

- `/etc/cas-release` says which image your Pi runs and when it was built.
- `10-cas.conf` holds the settings our image gives Ollama. The system starts Ollama at boot with these environment variables.
- `/opt/cas/scripts` holds the programs the course server can run, the same list as `caspi.scripts()`. `hello` is the program you ran in Section 4.6.4, a short shell script.

One command checks the whole setup at once. `cas-selfcheck` is a script on the image that reports the Pi's name and image, its network mode, temperature, power supply and free disk, and whether the servers your notebooks talk to answer. It ends with `ALL GOOD` or lists what is wrong. If your Pi misbehaves later in the course, run it first and send us its output:

```sh
sudo cas-selfcheck
```

#### 4.7.3 Ollama on the command line

Ollama also comes with a command-line program. It talks to the same Ollama server as `caspi.py`, so a model you loaded from the notebook appears here too, and the other way round.

- `ollama list` → list all available models
- `ollama ps` → list the models in memory
- `ollama show <MODEL>` → display model details (parameters, size, etc.)
- `ollama run <MODEL>` → start a chat with the model in the terminal. `/?` lists the chat commands and `/bye` ends the chat.
- `ollama stop <MODEL>` → drop the model from memory
- `--verbose` → print timing statistics after each answer, like caspi's line
- `--help` → explain any command, for example `ollama run --help`

Try, for example:

```sh
ollama run granite4.2:3b --verbose --think=false
```

#### 4.7.4 Watch the Pi work with htop

We can also monitor the Raspberry Pi's resource usage while it runs a model. There are many ways to do so, but `htop` is one of the quickest and simplest. It shows the processor cores, the memory and the running programs, updated live. The screenshot shows a machine with three cores; your Pi has four. Press `q` to quit `htop`.

<center>
<img src="../../.admin/notebooks/picture/htop.png"
     alt="Raspberry Pi 5: htop"
     width="60%">

Visit [htop Explained Visually](https://codeahoy.com/2017/01/20/hhtop-explained-visually/) to learn more.
</center>

<div class="alert alert-block alert-info">

1. Connect to the Pi via SSH and start `htop`.<br>
2. Place the terminal next to this notebook so that you can see both.<br>
3. Run the first cell below and watch the four core bars and the memory bar while the model writes.<br>
4. Run the second cell. It drops the model from memory and loads it again. Watch the memory bar while it loads.<br>
5. Optional: open a second terminal, log in again, and chat with `ollama run` while `htop` runs in the first one.
</div>

In [ ]:
caspi.ask(MODEL, "Write a short story, about 200 words, about a Raspberry Pi that learns to talk.", think=False)

In [ ]:
caspi.stop(MODEL)
caspi.load(MODEL)

### 4.8 Further Reading and Literature

[01] Raspberry Pi Ltd. **Raspberry Pi computer hardware.** Official documentation, including the Raspberry Pi 5. [Website](https://www.raspberrypi.com/documentation/computers/raspberry-pi.html)

[02] Ollama. **API documentation.** Every endpoint and option that `caspi.py` uses. [Documentation](https://docs.ollama.com/api)

[03] Mansoor, U. (2017). **htop Explained Visually.** CodeAhoy. [Article](https://codeahoy.com/2017/01/20/hhtop-explained-visually/)

---
### 4.9 Questions

<div class="alert alert-block alert-info">Fill in your responses and save this notebook. See the <a href="../../03_Assignments/Submission-Guidelines.md">submission guidelines</a> for how to submit your work.</div>

Use your own outputs as evidence. Copy the relevant numbers into your answer so that they remain visible after saving.

**Question 01: Connectors**

Get comfortable with the Raspberry Pi's connectors. Which are present? How many? What is each one for?

(answer here)

**Question 02: Cooling**

Does the Raspberry Pi get hot? Compare `caspi.health()["cpu_temp_c"]` at rest and right after a long answer. What are the two components that help the device stay cool? How do they work? What would happen without them?

*Hint: active vs passive*

(answer here)

**Question 03: Two reachability tests**

The ping in Section 4.7.1 and the HTTP request in Section 4.4.1 both reached the Pi. What does each test prove? Describe one situation in which the ping works but the HTTP request fails.

(answer here)

**Question 04: Models on disk and in memory**

How many models are available on your Pi, and how large is each file? Compare the size of your loaded model in `caspi.models(info=True)` with its `size` in `caspi.ps()`. Which is larger? Suggest what the difference is used for.

(answer here)

**Question 05: Tokens per second**

Take a closer look at `reply.raw`. Calculate the tokens per second for the prompt evaluation and for the response generation yourself, and compare your result with the statistics line. Which of the two is faster? Suggest why.

*Hint: visit the [API documentation](https://docs.ollama.com/api) to find out what the fields mean, in which unit the times are, and what `prompt_eval_cached_count` counts.*

(answer here)

**Question 06: Chat memory**

In Section 4.6.3, why does `chat` know your name while `ask` does not? Report the prompt tokens of the first and the second `chat` request, and how many of them were cached, and explain the difference. What does this mean for a long conversation on the Pi?

(answer here)

**Question 07: A small and a large model**

`qwen3.5:4b` is a file about two and a half times larger than `qwen3.5:0.8b`, from the same family. Without running anything: how do you expect the two to differ in speed and in the quality of their answers, and why? 05.09 measures both.

(answer here)

**Question 08: Ollama's settings**

Look at the Ollama settings you printed in Section 4.7.2. What does `OLLAMA_HOST=0.0.0.0:11434` do? What would happen to `caspi.py` without it? Pick one other setting in the file and explain what it changes.

*Hint: `ollama run --help` shows the default for `OLLAMA_HOST`.*

(answer here)

**Question 09: Watching the Pi**

Run a model and monitor the resource usage with `htop`: how high is the utilization across all cores during text generation? How high is it once the inference has completed? How much memory does your chosen model occupy once it has been loaded?

(answer here)

**Question 10: Local AI**

Everything in this notebook ran on a device smaller than a phone, with no internet connection. Where in your own organisation would that matter, and why? What do you give up compared with a large model in the cloud?

(answer here)

**OPTIONAL: Did you run into difficulties during the setup, and what helped? Did you read through and understand all the tasks in this Notebook? Are there any shortcomings, missing parts, or sections that should be improved?**

 (please help the team by telling us what part was not understood, and we will work on it)